# Sampled pool inspection

In [1]:
import bisect

import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

BIN_EDGES = [800, 1000, 1200, 1400, 1600, 1800, 2000, 2200]
BIN_NAMES = (["<800"]
             + [f"[{lo}, {hi})" for lo, hi in zip(BIN_EDGES[:-1], BIN_EDGES[1:])]
             + [">=2200"])
PER_BIN_TARGET = 50_000
MAX_ELO_DIFF = 100
MIN_HALF_MOVES = 20

df = pd.read_parquet("../data/splits/train.parquet")
df["elo_bin"] = pd.Categorical(df["elo_bin"], categories=BIN_NAMES, ordered=True)
g = df.groupby("elo_bin", observed=True)

print("shape:", df.shape)
print("columns: ", df.columns.tolist())
print("Data types:\n", df.dtypes)
display(df.drop(columns=["moves_san", "clocks_sec"]).head(3))

shape: (229402, 19)
columns:  ['game_id', 'white_elo', 'black_elo', 'avg_elo', 'rating_diff', 'time_control', 'result', 'termination', 'half_moves', 'moves_san', 'clocks_sec', 'elo_bin', 'white', 'black', 'utc_date', 'utc_time', 'eco', 'opening', 'split']
Data types:
 game_id              str
white_elo          int64
black_elo          int64
avg_elo          float64
rating_diff        int64
time_control         str
result               str
termination          str
half_moves         int64
moves_san            str
clocks_sec           str
elo_bin         category
white                str
black                str
utc_date             str
utc_time             str
eco                  str
opening              str
split                str
dtype: object


,game_id,white_elo,black_elo,avg_elo,rating_diff,time_control,result,termination,half_moves,elo_bin,white,black,utc_date,utc_time,eco,opening,split
0,LcvjuOZ0,789,810,799.5,-21,600+0,1-0,Normal,39,<800,furburn,nandokellom,2017.04.02,23:53:49,B06,Modern Defense,train
1,Jvo0bgaZ,794,804,799.0,-10,600+0,0-1,Normal,54,<800,nandokellom,kanchanfromindia,2017.04.04,11:54:27,A04,Zukertort Opening: Quiet System,train
2,TaaXdmGu,792,799,795.5,-7,600+0,1-0,Normal,35,<800,dmitry86rus,ormapajo,2017.04.07,01:46:43,B02,Alekhine Defense: Maroczy Variation,train


## Bin sizes

In [2]:
counts = df["elo_bin"].value_counts().sort_index()
display(pd.DataFrame({
    "n": counts,
    "share": (counts / len(df)).round(4),
    "at_cap": counts == PER_BIN_TARGET,
}))
print("above cap:", (counts > PER_BIN_TARGET).sum(), "bins")

,n,share,at_cap
elo_bin,,,
<800,35,0.0002,False
"[800, 1000)",14816,0.0646,False
"[1000, 1200)",34545,0.1506,False
"[1200, 1400)",34514,0.1505,False
"[1400, 1600)",34430,0.1501,False
"[1600, 1800)",34389,0.1499,False
"[1800, 2000)",33862,0.1476,False
"[2000, 2200)",35237,0.1536,False
>=2200,7574,0.0330,False


above cap: 0 bins


## Invariants

Every value below must be 0.

In [3]:
tc = df["time_control"].str.extract(r"^(\d+)\+(\d+)$").astype("Float64")
recomputed = df["avg_elo"].map(lambda x: BIN_NAMES[bisect.bisect_right(BIN_EDGES, x)])

checks = {
    "avg_elo mismatch":     (df["avg_elo"] != (df["white_elo"] + df["black_elo"]) / 2).sum(),
    "rating_diff mismatch": (df["rating_diff"] != df["white_elo"] - df["black_elo"]).sum(),
    "rating_diff too large": (df["rating_diff"].abs() > MAX_ELO_DIFF).sum(),
    "game too short":       (df["half_moves"] < MIN_HALF_MOVES).sum(),
    "half_moves mismatch":  ((df["moves_san"].str.count(",") + 1) != df["half_moves"]).sum(),
    "clocks incomplete":    df["clocks_sec"].str.contains(r"(?:^|,)(?:,|$)").sum(),
    "bad result":           (~df["result"].isin(["1-0", "0-1", "1/2-1/2"])).sum(),
    "bad termination":      df["termination"].isin(["Abandoned", "Rules infraction"]).sum(),
    "elo_bin mismatch":     (recomputed != df["elo_bin"].astype(str)).sum(),
    "duplicate game_id":    df["game_id"].duplicated().sum(),
    "time control unparsed": tc[0].isna().sum(),
    "outside rapid range":  ((tc[0] + 40 * tc[1] < 480) | (tc[0] + 40 * tc[1] > 1499)).sum(),
    "nan":                  df.isna().sum().sum(),
}
display(pd.Series(checks, name="violations").to_frame())

df["tc_base"] = tc[0]

,violations
avg_elo mismatch,0
rating_diff mismatch,0
rating_diff too large,0
game too short,0
half_moves mismatch,0
clocks incomplete,0
bad result,0
bad termination,0
elo_bin mismatch,0
duplicate game_id,0


## Per-bin characteristics

In [4]:
display(g[["avg_elo", "half_moves", "tc_base"]].median().round(1))

display(pd.crosstab(df["elo_bin"], df["result"], normalize="index").round(4))
display(pd.crosstab(df["elo_bin"], df["termination"], normalize="index").round(4))

top_tc = df["time_control"].value_counts().head(6).index
display(pd.crosstab(
    df["elo_bin"],
    df["time_control"].where(df["time_control"].isin(top_tc), "other"),
    normalize="index",
).round(3))

,avg_elo,half_moves,tc_base
elo_bin,,,
<800,794.0,51.0,600.0
"[800, 1000)",958.5,53.0,600.0
"[1000, 1200)",1137.0,56.0,600.0
"[1200, 1400)",1320.5,59.0,600.0
"[1400, 1600)",1505.5,63.0,600.0
"[1600, 1800)",1692.5,66.0,600.0
"[1800, 2000)",1871.5,69.0,600.0
"[2000, 2200)",2053.5,72.0,600.0
>=2200,2243.0,74.0,600.0


result,0-1,1-0,1/2-1/2
elo_bin,,,
<800,0.6000,0.3143,0.0857
"[800, 1000)",0.4692,0.4738,0.0570
"[1000, 1200)",0.4607,0.4892,0.0500
"[1200, 1400)",0.4596,0.4954,0.0451
"[1400, 1600)",0.4599,0.4963,0.0438
"[1600, 1800)",0.4634,0.4914,0.0452
"[1800, 2000)",0.4600,0.4847,0.0554
"[2000, 2200)",0.4474,0.4778,0.0748
>=2200,0.4315,0.4584,0.1101


termination,Normal,Time forfeit
elo_bin,,
<800,0.5143,0.4857
"[800, 1000)",0.7417,0.2583
"[1000, 1200)",0.8114,0.1886
"[1200, 1400)",0.8344,0.1656
"[1400, 1600)",0.8407,0.1593
"[1600, 1800)",0.8405,0.1595
"[1800, 2000)",0.8358,0.1642
"[2000, 2200)",0.8360,0.1640
>=2200,0.8449,0.1551


time_control,300+5,300+8,480+0,600+0,600+5,900+0,other
elo_bin,,,,,,,
<800,0.000,0.000,0.000,1.000,0.000,0.000,0.000
"[800, 1000)",0.001,0.001,0.002,0.955,0.004,0.015,0.024
"[1000, 1200)",0.005,0.006,0.006,0.875,0.006,0.026,0.076
"[1200, 1400)",0.012,0.020,0.014,0.761,0.016,0.051,0.126
"[1400, 1600)",0.032,0.040,0.030,0.648,0.017,0.065,0.169
"[1600, 1800)",0.044,0.035,0.043,0.612,0.021,0.063,0.182
"[1800, 2000)",0.056,0.030,0.057,0.599,0.019,0.061,0.177
"[2000, 2200)",0.085,0.023,0.072,0.516,0.031,0.055,0.218
>=2200,0.178,0.048,0.172,0.271,0.038,0.037,0.255


## Player Overlay

In [5]:
df = pd.read_parquet("../data/sampled/sampled_games_enriched.parquet")
df["elo_bin"] = pd.Categorical(df["elo_bin"], categories=BIN_NAMES, ordered=True)
df["date"] = pd.to_datetime(df["utc_date"] + " " + df["utc_time"],
                            format="%Y.%m.%d %H:%M:%S")

# Player overlap: how often does the same player appear across games?
players = pd.concat([df["white"], df["black"]])
vc = players.value_counts()
print("games:", len(df), "| distinct players:", len(vc))
print("share of players with a single game:", round((vc == 1).mean(), 3))
print("share of slots held by players with >=10 games:",
      round(vc[vc >= 10].sum() / vc.sum(), 3))
display(vc.describe().round(2).to_frame("games_per_player"))

# Same, per bin
long = pd.concat([
    df[["elo_bin", "white"]].rename(columns={"white": "player"}),
    df[["elo_bin", "black"]].rename(columns={"black": "player"}),
])
per_bin = long.groupby("elo_bin", observed=True)["player"].agg(
    slots="size", distinct="nunique")
per_bin["games_per_player"] = (per_bin["slots"] / per_bin["distinct"]).round(2)
display(per_bin)

# Month coverage: each bin should be spread across all three dumps
display(pd.crosstab(df["elo_bin"], df["date"].dt.to_period("M"),
                    normalize="index").round(3))

games: 333478 | distinct players: 91620
share of players with a single game: 0.313
share of slots held by players with >=10 games: 0.678


,games_per_player
count,91620.00
mean,7.28
std,14.08
min,1.00
25%,1.00
50%,3.00
75%,8.00
max,614.00


,slots,distinct,games_per_player
elo_bin,,,
<800,94,71,1.32
"[800, 1000)",44680,5288,8.45
"[1000, 1200)",100000,15534,6.44
"[1200, 1400)",100000,23919,4.18
"[1400, 1600)",100000,29245,3.42
"[1600, 1800)",100000,25911,3.86
"[1800, 2000)",100000,18282,5.47
"[2000, 2200)",100000,10205,9.80
>=2200,22182,2490,8.91


date,2017-03,2017-04,2017-05,2017-06
elo_bin,,,,
<800,0.000,0.298,0.447,0.255
"[800, 1000)",0.001,0.325,0.344,0.330
"[1000, 1200)",0.001,0.331,0.346,0.322
"[1200, 1400)",0.001,0.333,0.343,0.323
"[1400, 1600)",0.001,0.331,0.344,0.324
"[1600, 1800)",0.001,0.338,0.338,0.324
"[1800, 2000)",0.001,0.334,0.340,0.325
"[2000, 2200)",0.001,0.331,0.337,0.331
>=2200,0.000,0.344,0.337,0.319


In [6]:
long = pd.concat([
    df[["elo_bin", "white"]].rename(columns={"white": "player"}),
    df[["elo_bin", "black"]].rename(columns={"black": "player"}),
])
bins_per_player = long.groupby("player", observed=True)["elo_bin"].nunique()

print(bins_per_player.value_counts().sort_index())
print("share in >1 bin:", round((bins_per_player > 1).mean(), 3))

# how many game-slots belong to players spanning multiple bins
multi = set(bins_per_player[bins_per_player > 1].index)
print("share of slots from multi-bin players:",
      round(long["player"].isin(multi).mean(), 3))

# Elo range a single player covers
span = long.merge(
    pd.concat([df[["white", "avg_elo"]].rename(columns={"white": "player"}),
               df[["black", "avg_elo"]].rename(columns={"black": "player"})]),
    on="player", how="left")
rng = span.groupby("player")["avg_elo"].agg(lambda s: s.max() - s.min())
print(rng.describe().round(1))

elo_bin
1    55711
2    32699
3     3028
4      162
5       17
6        2
7        1
Name: count, dtype: int64
share in >1 bin: 0.392
share of slots from multi-bin players: 0.741
count    91620.0
mean        86.2
std         95.5
min          0.0
25%          0.0
50%         64.0
75%        141.0
max       1255.5
Name: avg_elo, dtype: float64
